In [1]:
!pip install -q datasets openai tqdm

In [1]:
from google.colab import userdata
import re
import json
import time
import random
from tqdm.notebook import tqdm
from datasets import load_dataset
from openai import OpenAI

In [2]:
# ── API settings ──────────────────────────────────────────
BASE_URL    = "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
API_KEY     = userdata.get('QWEN_API')
MODEL       = "qwen-plus"

# ── Sampling ──────────────────────────────────────────────
TRIAL_N     = 1319
SEED        = 42

# ── Generation ────────────────────────────────────────────
MAX_TOKENS  = 1024
TEMPERATURE = 0.0   # Greedy decoding — best for benchmarking
DELAY       = 0.0   # Seconds between requests (increase if rate-limited)

# ── Output ────────────────────────────────────────────────
OUTPUT_FILE = "qwen3_5_gsm8k_results.json"

# ── System prompt ─────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a helpful math assistant. "
    "Solve the problem step by step, then end your answer with: "
    "'The answer is <number>.'"
)

print("Config set!")
print(f"  Model   : {MODEL}")
print(f"  Trial N : {TRIAL_N or 'all 1319'}")
print(f"  Seed    : {SEED}")

Config set!
  Model   : qwen-plus
  Trial N : 1319
  Seed    : 42


In [3]:
def extract_answer(text: str) -> str | None:
    """Extract the final numeric answer from model output or gold label."""
    # GSM8K gold format: #### <number>
    match = re.search(r"####\s*([\d,.\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    # Model may say 'The answer is X'
    match = re.search(r"[Tt]he answer is[:\s]*([\d,.\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    # Fallback: last number in text
    numbers = re.findall(r"[\d,]+\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()

    return None


def is_correct(predicted: str | None, gold: str) -> bool:
    """Compare predicted and gold answers numerically."""
    if predicted is None:
        return False
    try:
        return abs(float(predicted) - float(gold)) < 1e-6
    except ValueError:
        return predicted.strip().lower() == gold.strip().lower()


def call_model(client: OpenAI, question: str) -> str:
    """Send a question to the model and return its response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
    )
    return response.choices[0].message.content or ""



In [4]:
print("Loading GSM8K test split ...")
full_dataset  = load_dataset("openai/gsm8k", "main", split="test")
full_size = len(full_dataset)  # 1319
print(full_size)

Loading GSM8K test split ...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

1319


In [5]:
if TRIAL_N is not None:
    if TRIAL_N > full_size:
        raise ValueError(f"TRIAL_N ({TRIAL_N}) exceeds full test set size ({full_size}).")
    random.seed(SEED)
    sampled_indices = sorted(random.sample(range(full_size), TRIAL_N))
    dataset = full_dataset.select(sampled_indices)
    print(f"Randomly sampled {TRIAL_N} of {full_size} examples (seed={SEED}).")
else:
    sampled_indices = list(range(full_size))
    print(f"Using full test set ({full_size} examples).")

Randomly sampled 1319 of 1319 examples (seed=42).


In [6]:
client  = OpenAI(base_url=BASE_URL, api_key=API_KEY)

total   = len(dataset)
correct = 0
errors  = 0
results = []

print(f"Running {total} examples ...\n")

for i, example in enumerate(tqdm(dataset, desc="Evaluating")):
    question = example["question"]
    gold_ans = extract_answer(example["answer"])  # parses #### <number>

    try:
        response     = call_model(client, question)
        pred_ans     = extract_answer(response)
        correct_flag = is_correct(pred_ans, gold_ans)
    except Exception as exc:
        response     = f"ERROR: {exc}"
        pred_ans     = None
        correct_flag = False
        errors      += 1

    if correct_flag:
        correct += 1

    results.append({
        "id":          i,
        "dataset_idx": sampled_indices[i],
        "question":    question,
        "gold":        gold_ans,
        "predicted":   pred_ans,
        "correct":     correct_flag,
        "response":    response,
    })

    if DELAY > 0:
        time.sleep(DELAY)

accuracy = correct / total * 100

summary = {
    "model":    MODEL,
    "trial_n":  TRIAL_N,
    "seed":     SEED,
    "total":    total,
    "correct":  correct,
    "errors":   errors,
    "accuracy": round(accuracy, 2),
}

print("\n" + "=" * 45)
print(f"  Model    : {summary['model']}")
print(f"  Trial N  : {summary['trial_n'] or 'all'}")
print(f"  Seed     : {summary['seed']}")
print(f"  Total    : {summary['total']}")
print(f"  Correct  : {summary['correct']}")
print(f"  Errors   : {summary['errors']}")
print(f"  Accuracy : {summary['accuracy']}%")
print("=" * 45)

Running 1319 examples ...



Evaluating:   0%|          | 0/1319 [00:00<?, ?it/s]


  Model    : qwen-plus
  Trial N  : 1319
  Seed     : 42
  Total    : 1319
  Correct  : 1241
  Errors   : 0
  Accuracy : 94.09%


In [7]:
output = {"summary": summary, "results": results}

with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to: {OUTPUT_FILE}")

# Download the file directly in Colab
from google.colab import files
files.download(OUTPUT_FILE)

Results saved to: qwen3_5_gsm8k_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
failures = [r for r in results if not r["correct"]]
print(f"Wrong answers: {len(failures)} / {total}\n")

# Print first 3 failures for a quick sanity check
for r in failures[:3]:
    print(f"--- Example {r['dataset_idx']} ---")
    print(f"Question  : {r['question']}...")
    print(f"Gold      : {r['gold']}")
    print(f"Predicted : {r['predicted']}")
    print(f"Response  : {r['response']}...")
    print()

Wrong answers: 78 / 1319

--- Example 40 ---
Question  : Brandon's iPhone is four times as old as Ben's iPhone. Ben's iPhone is two times older than Suzy's iPhone. If Suzy’s iPhone is 1 year old, how old is Brandon’s iPhone?...
Gold      : 8
Predicted : 12.
Response  : We are given:

- Suzy’s iPhone is **1 year old**.
- Ben's iPhone is **two times older** than Suzy's iPhone.
- Brandon's iPhone is **four times as old** as Ben's iPhone.

Let’s interpret carefully.

### Step 1: Ben's iPhone age

"Ben's iPhone is two times older than Suzy's iPhone."

⚠️ Note: The phrase *"two times older than"* is often ambiguous, but in standard mathematical usage, it usually means:

> Ben's age = Suzy's age + 2 × Suzy's age = 3 × Suzy's age.

But many people (especially in casual speech) use *"two times older than"* to mean *"two times as old as"*, i.e., just 2×.

However, let's examine both interpretations and see which makes sense contextually — and check for consistency.

#### Interpretation A: "two t